# K-Prototypes Clustering Across Datasets

Python/Zerve notebook version of `Clustering/k_prototype.R`.

This notebook applies k-prototypes clustering to the complete-case baseline and the V2 imputed datasets:

- complete-case baseline
- mean/mode imputation
- KNN imputation, k = 5
- KNN imputation, k = 10
- MICE imputation 1
- MICE imputation 2

It uses `kmodes.kprototypes.KPrototypes`, the closest Python counterpart to R's `clustMixType::kproto`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from kmodes.kprototypes import KPrototypes
from sklearn.metrics import adjusted_rand_score, pairwise_distances, silhouette_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 123

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Clustering" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "Datasets" / "V2"
OUTPUT_DIR = PROJECT_ROOT / "Clustering" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## 1. Load V2 Datasets

In [ ]:
dataset_paths = {
    "baseline": DATA_DIR / "data_complete_baseline.csv",
    "mean_mode": DATA_DIR / "mean_mode_imputed_dataset.csv",
    "knn_k5": DATA_DIR / "knn_imputed_data_k5.csv",
    "knn_k10": DATA_DIR / "knn_imputed_data_k10.csv",
    "mice1": DATA_DIR / "mice_imputation_data1.csv",
    "mice2": DATA_DIR / "mice_imputation_data2.csv",
}

datasets = {name: pd.read_csv(path) for name, path in dataset_paths.items()}

for name, df in datasets.items():
    print(f"{name:10s} {df.shape}")

## 2. Shared Preprocessing

The R script converts state to Census-style region, treats age/income/religious attendance as ordered categories, keeps survey response columns as categorical variables, and coerces amount/multiplier/political/religious views to numeric.

In [ ]:
STATE_TO_REGION = {
    **dict.fromkeys(["CT", "ME", "MA", "NH", "RI", "VT", "NJ", "NY", "PA"], "Northeast"),
    **dict.fromkeys(["IL", "IN", "IA", "KS", "MI", "MN", "MO", "NE", "ND", "OH", "SD", "WI"], "Midwest"),
    **dict.fromkeys(["AL", "AR", "DE", "DC", "FL", "GA", "KY", "LA", "MD", "MS", "NC", "OK", "SC", "TN", "TX", "VA", "WV"], "South"),
    **dict.fromkeys(["AK", "AZ", "CA", "CO", "HI", "ID", "MT", "NV", "NM", "OR", "UT", "WA", "WY", "0"], "West"),
}

AGE_LEVELS = ["18-29", "30-39", "40-49", "50-59", "60-69", "70 or over"]
INCOME_LEVELS = ["Under $20,000", "$20,000 - $39,999", "$40,000 - $59,999", "$60,000 - $79,999", "$80,000 - $99,999", "Over $100,000"]
ATTENDANCE_LEVELS = ["Never", "Seldom", "A few times a year", "Once or twice a month", "Once a week", "More than once a week"]

STATE_COL = "Q9.What.State.do.you.live.in."
VOTE_COL = "Q5.In.the.2016.Presidential.election..who.did.you.vote.for."
PARTY_COL = "Q7.Do.you.consider.yourself.a."
ATTEND_COL = "Q8.Aside.from.weddings.and.funerals..how.often.do.you.attend.religious.services."
POL_COL = "Q3_1.On.a.scale.of.0.to.100..how.would.you.describe.your.political.views."
REL_COL = "Q8_1.On.a.scale.of.0.to.100..how.would.you.describe.your.religious.orientation."

NUMERIC_COLS = ["multiplier", "amount", POL_COL, REL_COL]
CATEGORICAL_COLS = ["age", "income", "gender", VOTE_COL, STATE_COL, PARTY_COL, ATTEND_COL, "batch"]

AGE_MAP = {v: i + 1 for i, v in enumerate(AGE_LEVELS)}
INCOME_MAP = {v: i + 1 for i, v in enumerate(INCOME_LEVELS)}
ATTEND_MAP = {v: i + 1 for i, v in enumerate(ATTENDANCE_LEVELS)}

def prepare_kproto_data(df):
    out = df.copy()
    out[STATE_COL] = out[STATE_COL].astype(str).str.strip().str.upper().map(STATE_TO_REGION)
    for col in NUMERIC_COLS:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out[NUMERIC_COLS + CATEGORICAL_COLS].dropna().copy()
    for col in CATEGORICAL_COLS:
        out[col] = out[col].astype(str)
    return out

def kproto_matrix(df):
    matrix = df[NUMERIC_COLS + CATEGORICAL_COLS].copy()
    return matrix.to_numpy(dtype=object), list(range(len(NUMERIC_COLS), len(NUMERIC_COLS) + len(CATEGORICAL_COLS)))

def fit_kproto(df, k=2, n_init=5):
    matrix, categorical_idx = kproto_matrix(df)
    model = KPrototypes(n_clusters=k, init="Cao", n_init=n_init, random_state=RANDOM_STATE, verbose=0)
    labels = model.fit_predict(matrix, categorical=categorical_idx)
    return model, labels

prepared = {name: prepare_kproto_data(df) for name, df in datasets.items()}
for name, df in prepared.items():
    print(f"{name:10s} {df.shape}")

## 3. Elbow and Silhouette Helpers

In [ ]:
def gower_distance_matrix(df, numeric_cols=NUMERIC_COLS, categorical_cols=CATEGORICAL_COLS):
    numeric = df[numeric_cols].astype(float).copy()
    ranges = numeric.max() - numeric.min()
    ranges = ranges.replace(0, 1)
    numeric_scaled = (numeric - numeric.min()) / ranges
    numeric_dist = pairwise_distances(numeric_scaled, metric="manhattan") / len(numeric_cols)

    cat = df[categorical_cols].astype(str)
    cat_dist = np.zeros((len(df), len(df)))
    for col in categorical_cols:
        values = cat[col].to_numpy()
        cat_dist += (values[:, None] != values[None, :]).astype(float)
    cat_dist = cat_dist / len(categorical_cols)

    return (numeric_dist * len(numeric_cols) + cat_dist * len(categorical_cols)) / (len(numeric_cols) + len(categorical_cols))

def elbow_table(df, max_k=10):
    rows = []
    for k in range(1, max_k + 1):
        model, labels = fit_kproto(df, k=k, n_init=3)
        rows.append({"k": k, "cost": model.cost_})
    return pd.DataFrame(rows)

def silhouette_table(df, k_values=range(2, 7)):
    dist = gower_distance_matrix(df)
    rows = []
    for k in k_values:
        _, labels = fit_kproto(df, k=k, n_init=3)
        rows.append({"k": k, "silhouette": silhouette_score(dist, labels, metric="precomputed")})
    return pd.DataFrame(rows)

def plot_metric(table, dataset_name, y_col, title):
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.lineplot(data=table, x="k", y=y_col, marker="o", ax=ax)
    ax.set_title(f"{title}: {dataset_name}")
    ax.set_xlabel("Number of clusters (k)")
    plt.show()

## 4. Baseline K-Prototypes

In [ ]:
baseline_df = prepared["baseline"]

baseline_elbow = elbow_table(baseline_df)
display(baseline_elbow)
plot_metric(baseline_elbow, "baseline", "cost", "K-Prototypes Elbow")

baseline_sil = silhouette_table(baseline_df)
display(baseline_sil)
plot_metric(baseline_sil, "baseline", "silhouette", "Gower Silhouette")

baseline_model, baseline_labels = fit_kproto(baseline_df, k=2)
print(pd.Series(baseline_labels).value_counts().sort_index())
print("Cost:", baseline_model.cost_)

## 5. Run K-Prototypes Across All Datasets

In [ ]:
all_results = {}

for name, df in prepared.items():
    print(f"Running {name}...")
    model, labels = fit_kproto(df, k=2)
    all_results[name] = {
        "model": model,
        "labels": labels,
        "cost": model.cost_,
        "sizes": pd.Series(labels).value_counts().sort_index(),
    }

summary = pd.DataFrame({
    name: {"n": len(prepared[name]), "cost": result["cost"], "cluster_0": result["sizes"].get(0, 0), "cluster_1": result["sizes"].get(1, 0)}
    for name, result in all_results.items()
}).T

display(summary)

## 6. Elbow and Silhouette for Imputed Datasets

This is the compact Python version of the repeated blocks in the R script.

In [ ]:
diagnostics = {}
for name in ["mean_mode", "knn_k5", "knn_k10", "mice1", "mice2"]:
    df = prepared[name]
    e = elbow_table(df)
    s = silhouette_table(df)
    diagnostics[name] = {"elbow": e, "silhouette": s}
    print(name)
    display(e)
    display(s)
    plot_metric(e, name, "cost", "K-Prototypes Elbow")
    plot_metric(s, name, "silhouette", "Gower Silhouette")

## 7. ARI Comparison Across Imputed Datasets

In [ ]:
compare_names = ["mean_mode", "knn_k5", "knn_k10", "mice1", "mice2"]
labels_for_ari = {name: all_results[name]["labels"] for name in compare_names}

ari_matrix = pd.DataFrame(index=compare_names, columns=compare_names, dtype=float)
for a in compare_names:
    for b in compare_names:
        # All V2 imputed datasets should have the same rows/order.
        ari_matrix.loc[a, b] = adjusted_rand_score(labels_for_ari[a], labels_for_ari[b])

display(ari_matrix.round(4))

## 8. Cluster Profiles

The profile mirrors the final summary part of `k_prototype.R`: outcome amount, giving behavior, multiplier, political/religious scales, and selected demographic proportions.

In [ ]:
def cluster_profile(original_df, prepared_df, labels):
    aligned = original_df.loc[prepared_df.index].copy()
    aligned["cluster"] = labels

    def prop_equals(series, value):
        return series.astype(str).eq(value).mean()

    rows = []
    for cluster_id, g in aligned.groupby("cluster"):
        rows.append({
            "cluster": cluster_id,
            "n": len(g),
            "avg_amount": pd.to_numeric(g["amount"], errors="coerce").mean(),
            "prop_give": (pd.to_numeric(g["amount"], errors="coerce") > 0).mean(),
            "avg_multiplier": pd.to_numeric(g["multiplier"], errors="coerce").mean(),
            "avg_pol_views": pd.to_numeric(g[POL_COL], errors="coerce").mean(),
            "avg_relig_views": pd.to_numeric(g[REL_COL], errors="coerce").mean(),
            "avg_age_level": g["age"].map(AGE_MAP).mean(),
            "avg_income_level": g["income"].map(INCOME_MAP).mean(),
            "avg_relig_attend_level": g[ATTEND_COL].map(ATTEND_MAP).mean(),
            "prop_female": prop_equals(g["gender"], "Female"),
            "prop_voted_trump": g[VOTE_COL].astype(str).str.contains("Trump", case=False, na=False).mean(),
            "prop_voted_clinton": g[VOTE_COL].astype(str).str.contains("Clinton", case=False, na=False).mean(),
            "prop_democrat": g[PARTY_COL].astype(str).str.contains("Democrat", case=False, na=False).mean(),
            "prop_republican": g[PARTY_COL].astype(str).str.contains("Republican", case=False, na=False).mean(),
        })
    return pd.DataFrame(rows).round(3)

profiles = {}
for name in ["baseline", "mean_mode", "knn_k5", "knn_k10", "mice1", "mice2"]:
    profiles[name] = cluster_profile(datasets[name], prepared[name], all_results[name]["labels"])
    print(name)
    display(profiles[name])

## 9. K-Prototypes Cluster Visualization

These plots compare the two k-prototypes clusters within each dataset using four interpretation variables: donation amount, religious views, age level, and Trump vote proportion.

For interpretability, cluster labels are re-oriented separately for each dataset so that **Cluster 1 has the higher average donation amount**. The heatmap shows `Cluster 1 average - Cluster 2 average`, so positive values mean Cluster 1 is higher.


In [ ]:
DATASET_DISPLAY_ORDER = ["baseline", "knn_k5", "knn_k10", "mean_mode", "mice1", "mice2"]
DATASET_DISPLAY_LABELS = {
    "baseline": "Baseline",
    "knn_k5": "KNN k=5",
    "knn_k10": "KNN k=10",
    "mean_mode": "Mean/Mode",
    "mice1": "MICE 1",
    "mice2": "MICE 2",
}

PROFILE_PLOT_VARIABLES = [
    "avg_amount",
    "avg_relig_views",
    "avg_age_level",
    "prop_voted_trump",
]

PROFILE_LABELS = {
    "avg_amount": "Amount",
    "avg_relig_views": "Religious views",
    "avg_age_level": "Age level",
    "prop_voted_trump": "Trump vote prop.",
}

def orient_profile_clusters_by_amount(profile):
    profile = profile.copy()
    high_amount_raw = profile.loc[profile["avg_amount"].idxmax(), "cluster"]
    profile["Cluster"] = np.where(profile["cluster"] == high_amount_raw, "Cluster 1", "Cluster 2")
    return profile

oriented_profiles = []
for dataset_name in DATASET_DISPLAY_ORDER:
    profile = orient_profile_clusters_by_amount(profiles[dataset_name])
    profile["dataset"] = DATASET_DISPLAY_LABELS[dataset_name]
    profile["dataset_key"] = dataset_name
    oriented_profiles.append(profile)

oriented_profiles = pd.concat(oriented_profiles, ignore_index=True)
profile_plot_long = oriented_profiles.melt(
    id_vars=["dataset", "dataset_key", "Cluster"],
    value_vars=PROFILE_PLOT_VARIABLES,
    var_name="variable",
    value_name="value",
)
profile_plot_long["variable_label"] = profile_plot_long["variable"].map(PROFILE_LABELS)
profile_plot_long["dataset"] = pd.Categorical(
    profile_plot_long["dataset"],
    categories=[DATASET_DISPLAY_LABELS[x] for x in DATASET_DISPLAY_ORDER],
    ordered=True,
)
profile_plot_long["Cluster"] = pd.Categorical(
    profile_plot_long["Cluster"],
    categories=["Cluster 1", "Cluster 2"],
    ordered=True,
)

diff_table = (
    profile_plot_long
    .pivot_table(index=["dataset", "variable", "variable_label"], columns="Cluster", values="value", observed=False)
    .reset_index()
)
diff_table["difference"] = diff_table["Cluster 1"] - diff_table["Cluster 2"]
diff_matrix = diff_table.pivot(index="dataset", columns="variable_label", values="difference")
diff_matrix = diff_matrix[[PROFILE_LABELS[v] for v in PROFILE_PLOT_VARIABLES]]

plt.figure(figsize=(9.5, 4.8))
sns.heatmap(
    diff_matrix,
    center=0,
    cmap="RdYlGn",
    annot=True,
    fmt=".2f",
    linewidths=0.4,
    cbar_kws={"label": "Cluster 1 average - Cluster 2 average"},
)
plt.title("K-Prototypes: Selected Cluster Differences")
plt.xlabel("")
plt.ylabel("")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(OUTPUT_DIR / "kprototype_selected_cluster_difference_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

grid = sns.catplot(
    data=profile_plot_long,
    x="dataset",
    y="value",
    hue="Cluster",
    col="variable_label",
    col_wrap=2,
    kind="bar",
    sharey=False,
    height=3.3,
    aspect=1.25,
    palette={"Cluster 1": "#D2691E", "Cluster 2": "#4C9F70"},
)
grid.set_titles("{col_name}")
grid.set_axis_labels("", "Average / proportion")
grid.fig.suptitle("K-Prototypes: Two-Cluster Comparison", y=1.03)
for ax in grid.axes.flatten():
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
grid.fig.savefig(OUTPUT_DIR / "kprototype_selected_cluster_average_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. Save Key Outputs

In [ ]:
summary.to_csv(OUTPUT_DIR / "kprototype_dataset_summary.csv")
ari_matrix.to_csv(OUTPUT_DIR / "kprototype_imputed_ari_matrix.csv")
for name, profile in profiles.items():
    profile.to_csv(OUTPUT_DIR / f"kprototype_profile_{name}.csv", index=False)

print(f"Saved outputs to: {OUTPUT_DIR}")